# KAUST Kaggle Challenge 3 — Stage 4
## Robust Neighbour-Width Optimization

This notebook refines the strongest result from the previous experiments rather than changing model families.

**Fixed configuration**
- CatBoost
- no calendar/date features
- no Temperature / Relative_Humidity
- raw S1/S2 + sensor-relationship + neighbour + existing rolling-context features
- threshold fixed at **0.5**
- minute-level timestamp precision for both train and test

**Only experiment:** neighbour width `k ∈ {3, 5, 7, 9}`.

Primary selection favors the width that is strong under both forward validation and the strict Nov 24–30 holdout. The selected width is then stress-tested across three arbitrary within-minute orderings and used to create an order-averaged submission.

Nothing is submitted automatically.

In [ ]:
import os, sys, glob, json, time, gc, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 100)

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
os.makedirs(WORK, exist_ok=True)
T0 = time.time()

try:
    from catboost import CatBoostClassifier
    import catboost
    print("catboost", catboost.__version__)
except ImportError:
    print("catboost missing; installing...")
    os.system(f"{sys.executable} -m pip install -q catboost")
    from catboost import CatBoostClassifier

print("pandas", pd.__version__, "| numpy", np.__version__)
print("working directory:", WORK)

In [ ]:
# Locate only the competition CSVs.
def find_competition_files():
    search_roots = ["/kaggle/input", "/mnt/data", ".", ".."]
    for root in search_roots:
        if not os.path.isdir(root):
            continue

        train_paths = glob.glob(os.path.join(root, "**", "train.csv"), recursive=True)
        test_paths = glob.glob(os.path.join(root, "**", "test.csv"), recursive=True)
        sample_paths = glob.glob(os.path.join(root, "**", "sample_submission.csv"), recursive=True)

        if train_paths and test_paths:
            return {
                "train": train_paths[0],
                "test": test_paths[0],
                "sample": sample_paths[0] if sample_paths else None,
            }

    raise FileNotFoundError("Could not locate train.csv and test.csv")

FILES = find_competition_files()
print(json.dumps(FILES, indent=2))

train = pd.read_csv(FILES["train"])
test = pd.read_csv(FILES["test"])
sample = pd.read_csv(FILES["sample"]) if FILES["sample"] else None

print("train:", train.shape)
print("test :", test.shape)
if sample is not None:
    print("sample:", sample.shape)

In [ ]:
def parse_datetime(series):
    s = series.astype(str)
    for fmt in ("%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M", "%Y-%m-%d %H:%M"):
        out = pd.to_datetime(s, format=fmt, errors="coerce")
        if out.notna().all():
            return out, fmt

    out = pd.to_datetime(s, errors="coerce", dayfirst=False)
    if out.isna().any():
        raise ValueError(f"{out.isna().sum()} timestamps failed to parse")
    return out, "fallback"

train["dt"], fmt_train = parse_datetime(train["Datetime"])
test["dt"], fmt_test = parse_datetime(test["Datetime"])

TARGET = "Offset_fault"
S1 = "Sensor1_PM2.5"
S2 = "Sensor2_PM2.5"
EPS = 1e-3

assert TARGET in train.columns
assert "ID" in test.columns
assert train["dt"].max() < test["dt"].min()
assert test["dt"].between(pd.Timestamp("2021-11-30"), pd.Timestamp("2022-02-01")).all()

print("train datetime:", fmt_train, train["dt"].min(), "->", train["dt"].max())
print("test datetime :", fmt_test, test["dt"].min(), "->", test["dt"].max())
print("train positive rate:", round(train[TARGET].mean(), 6))
print("seconds==0 fraction | train:", round((train["dt"].dt.second == 0).mean(), 5),
      "| test:", round((test["dt"].dt.second == 0).mean(), 5))

## Feature construction

The full provided observation stream is used only for **target-independent** local context. No test labels exist or are used.

Both train and test timestamps are floored to the minute. Observations sharing a minute receive a deterministic pseudo-random tie order controlled by `order_seed`.

The feature set deliberately excludes:
- explicit date/calendar variables;
- Temperature;
- Relative_Humidity.

In [ ]:
def build_stream(order_seed=0):
    a = train.copy()
    b = test.copy()

    a["_orig_train_idx"] = np.arange(len(a), dtype=np.int64)
    a["_orig_test_id"] = -1
    a["is_test"] = 0
    a["ID"] = -1

    b["_orig_train_idx"] = -1
    b["_orig_test_id"] = b["ID"].astype(np.int64)
    b["is_test"] = 1
    b[TARGET] = np.nan

    full = pd.concat([a, b], ignore_index=True, sort=False)
    full["tmin"] = full["dt"].dt.floor("min")

    rng = np.random.RandomState(order_seed)
    full["_tie"] = rng.random(len(full))
    full = full.sort_values(["tmin", "_tie"]).reset_index(drop=True)
    return full


def add_relation_features(f, full):
    diff = full[S1] - full[S2]
    f["S1"] = full[S1]
    f["S2"] = full[S2]
    f["diff"] = diff
    f["abs_diff"] = diff.abs()
    f["ratio"] = full[S1] / (full[S2].abs() + EPS)
    f["rel_diff"] = diff / (full[S2].abs() + EPS)
    f["sym_rel"] = 2.0 * diff / (full[S1].abs() + full[S2].abs() + EPS)
    f["log_ratio"] = np.log(full[S1].clip(lower=0.01)) - np.log(full[S2].clip(lower=0.01))
    f["level"] = (full[S1] + full[S2]) / 2.0
    f["resid_line"] = full[S1] - (1.0039 * full[S2] + 0.7551)

    return [
        "S1", "S2", "diff", "abs_diff", "ratio", "rel_diff",
        "sym_rel", "log_ratio", "level", "resid_line"
    ]


def add_neighbour_features(f, full, k):
    names = []
    ratio = full[S1] / (full[S2].abs() + EPS)

    for col, nm in ((S1, "S1"), (S2, "S2"), (None, "ratio")):
        base = ratio if col is None else full[col]
        sides = {}

        for j in range(1, k + 1):
            sides[f"m{j}"] = base.shift(j)
            sides[f"p{j}"] = base.shift(-j)

            # Keep explicit raw lag/lead values only for S1/S2, matching prior notebooks.
            if col is not None:
                f[f"{nm}_lag{j}"] = sides[f"m{j}"]
                f[f"{nm}_lead{j}"] = sides[f"p{j}"]
                names += [f"{nm}_lag{j}", f"{nm}_lead{j}"]

        N = pd.DataFrame(sides)
        med = N.median(axis=1)
        mean = N.mean(axis=1)
        sd = N.std(axis=1)

        f[f"{nm}_nb_med"] = med
        f[f"{nm}_nb_std"] = sd
        f[f"{nm}_nb_dev"] = base - med
        f[f"{nm}_nb_ratio"] = base / (med.abs() + EPS)
        f[f"{nm}_nb_z"] = (base - mean) / (sd + EPS)

        names += [
            f"{nm}_nb_med", f"{nm}_nb_std", f"{nm}_nb_dev",
            f"{nm}_nb_ratio", f"{nm}_nb_z"
        ]

    f["nb_ratio_gap"] = f["S1_nb_ratio"] / (f["S2_nb_ratio"].abs() + EPS)
    f["nb_dev_gap"] = f["S1_nb_dev"] - f["S2_nb_dev"]
    f["gap_prev_s"] = (full["tmin"] - full["tmin"].shift(1)).dt.total_seconds()
    f["gap_next_s"] = (full["tmin"].shift(-1) - full["tmin"]).dt.total_seconds()

    names += ["nb_ratio_gap", "nb_dev_gap", "gap_prev_s", "gap_next_s"]
    return names


def add_rolling_features(f, full, windows=("30min", "360min")):
    # Fixed rolling family from the prior best configuration.
    names = []
    g = full.set_index("tmin")
    ratio_vals = (full[S1] / (full[S2].abs() + EPS)).values

    for w in windows:
        for col, nm in ((S1, "S1"), (S2, "S2")):
            med = g[col].rolling(w, center=True, closed="both").median().values
            f[f"{nm}_roll_ratio_{w}"] = full[col].values / (np.abs(med) + EPS)
            f[f"{nm}_roll_dev_{w}"] = full[col].values - med
            names += [f"{nm}_roll_ratio_{w}", f"{nm}_roll_dev_{w}"]

        rmed = (g[S1] / (g[S2].abs() + EPS)).rolling(
            w, center=True, closed="both"
        ).median().values
        f[f"ratio_roll_dev_{w}"] = ratio_vals - rmed
        names.append(f"ratio_roll_dev_{w}")

    return names


def build_dataset(k, order_seed=0):
    full = build_stream(order_seed=order_seed)
    feat = pd.DataFrame(index=full.index)

    relation_cols = add_relation_features(feat, full)
    neighbour_cols = add_neighbour_features(feat, full, k=k)
    rolling_cols = add_rolling_features(feat, full)

    cols = relation_cols + neighbour_cols + rolling_cols
    feat = feat.replace([np.inf, -np.inf], np.nan).astype(np.float32)

    m_train = (full["is_test"] == 0).values

    X = feat.loc[m_train, cols].reset_index(drop=True)
    y = full.loc[m_train, TARGET].astype(int).to_numpy()
    dt = full.loc[m_train, "dt"].reset_index(drop=True)
    train_key = full.loc[m_train, "_orig_train_idx"].astype(np.int64).to_numpy()

    Xt = feat.loc[~m_train, cols].reset_index(drop=True)
    test_ids = full.loc[~m_train, "_orig_test_id"].astype(np.int64).to_numpy()

    assert len(X) == len(train)
    assert len(Xt) == len(test)
    assert len(np.unique(train_key)) == len(train)
    assert len(np.unique(test_ids)) == len(test)

    return full, X, y, dt, train_key, Xt, test_ids, cols

## Frozen model and validation

Threshold is fixed at **0.5**. No threshold tuning is used in Stage 4.

Selection prioritizes robustness:
`robust_score = min(forward_mean_F1, strict_holdout_F1)`.

In [ ]:
CB_PARAMS = dict(
    iterations=600,
    learning_rate=0.08,
    depth=6,
    l2_leaf_reg=3.0,
    loss_function="Logloss",
    eval_metric="Logloss",
    random_seed=SEED,
    thread_count=-1,
    verbose=0,
    allow_writing_files=False,
)

FOLD_CUTS = [
    ("2021-10-15", "2021-11-02", "2021-11-09"),
    ("2021-10-15", "2021-11-09", "2021-11-16"),
    ("2021-10-15", "2021-11-16", "2021-11-23"),
    ("2021-10-15", "2021-11-23", "2021-12-01"),
]

HOLDOUT_START = pd.Timestamp("2021-11-24")
THRESHOLD = 0.5
K_VALUES = [3, 5, 7, 9]

print("CatBoost:", CB_PARAMS)
print("k values:", K_VALUES)
print("threshold:", THRESHOLD)


def fit_predict(Xtr, ytr, Xva):
    model = CatBoostClassifier(**CB_PARAMS)
    model.fit(Xtr, ytr)
    proba = model.predict_proba(Xva)[:, 1]
    return model, proba


def report_binary(y_true, proba, threshold=THRESHOLD):
    pred = (np.asarray(proba) >= threshold).astype(np.int8)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, pred, average="binary", zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "f1": float(f),
        "precision": float(p),
        "recall": float(r),
        "positive_rate": float(pred.mean()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

In [ ]:
# Sweep neighbour widths under the exact same no-weather/no-calendar setup.
SUMMARY_ROWS = []
FOLD_ROWS = []

for k in K_VALUES:
    tk = time.time()
    print("\n" + "=" * 80)
    print(f"k = {k}")
    print("=" * 80)

    full_k, X, y, dt, train_key, Xt, test_ids, cols = build_dataset(k=k, order_seed=0)
    print("features:", len(cols), "| train matrix:", X.shape)

    fold_f1 = []

    for fold, (start, mid, end) in enumerate(FOLD_CUTS):
        itr = np.flatnonzero((dt >= pd.Timestamp(start)) & (dt < pd.Timestamp(mid)))
        iva = np.flatnonzero((dt >= pd.Timestamp(mid)) & (dt < pd.Timestamp(end)))

        _, p = fit_predict(X.iloc[itr], y[itr], X.iloc[iva])
        rep = report_binary(y[iva], p)

        fold_f1.append(rep["f1"])
        FOLD_ROWS.append({
            "k": k,
            "fold": fold,
            "val_start": mid,
            "val_end": end,
            "n_train": len(itr),
            "n_val": len(iva),
            **rep,
        })

        print(
            f"fold {fold}: F1={rep['f1']:.6f} "
            f"P={rep['precision']:.5f} R={rep['recall']:.5f} "
            f"pos={rep['positive_rate']:.5f}"
        )

    i_pre = np.flatnonzero(dt < HOLDOUT_START)
    i_hold = np.flatnonzero(dt >= HOLDOUT_START)

    _, p_hold = fit_predict(X.iloc[i_pre], y[i_pre], X.iloc[i_hold])
    hold = report_binary(y[i_hold], p_hold)

    forward_mean = float(np.mean(fold_f1))
    forward_std = float(np.std(fold_f1))
    forward_min = float(np.min(fold_f1))
    robust_score = float(min(forward_mean, hold["f1"]))

    row = {
        "k": k,
        "n_features": len(cols),
        "forward_f1_mean": forward_mean,
        "forward_f1_std": forward_std,
        "forward_f1_min": forward_min,
        "holdout_f1": hold["f1"],
        "holdout_precision": hold["precision"],
        "holdout_recall": hold["recall"],
        "holdout_positive_rate": hold["positive_rate"],
        "robust_score": robust_score,
        "runtime_min": (time.time() - tk) / 60.0,
    }
    SUMMARY_ROWS.append(row)

    print(
        f"STRICT HOLDOUT: F1={hold['f1']:.6f} "
        f"P={hold['precision']:.5f} R={hold['recall']:.5f} "
        f"pos={hold['positive_rate']:.5f}"
    )
    print(
        f"forward mean={forward_mean:.6f} ± {forward_std:.6f} "
        f"| worst={forward_min:.6f} | robust_score={robust_score:.6f}"
    )

    del full_k, X, Xt
    gc.collect()

summary_df = pd.DataFrame(SUMMARY_ROWS)
fold_df = pd.DataFrame(FOLD_ROWS)

summary_df = summary_df.sort_values(
    ["robust_score", "holdout_f1", "forward_f1_mean", "forward_f1_std"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

summary_path = os.path.join(WORK, "stage4_neighbor_width_metrics.csv")
fold_path = os.path.join(WORK, "stage4_fold_metrics.csv")
summary_df.to_csv(summary_path, index=False)
fold_df.to_csv(fold_path, index=False)

print("\nFINAL WIDTH COMPARISON")
print(summary_df.round(6).to_string(index=False))
print("\nsaved:", summary_path)
print("saved:", fold_path)

In [ ]:
BEST_K = int(summary_df.iloc[0]["k"])
BEST_ROBUST_SCORE = float(summary_df.iloc[0]["robust_score"])

print("Selected BEST_K =", BEST_K)
print("Selection rule: max min(forward_mean_F1, strict_holdout_F1)")
print("best robust score:", round(BEST_ROBUST_SCORE, 7))

## Within-minute ordering robustness

The selected `k` is rebuilt with three different deterministic tie orders for observations that share the same minute.

The holdout probabilities are aligned by original training-row identity before averaging.

In [ ]:
ORDER_SEEDS = [0, 1, 2]
ORDER_ROWS = []
holdout_prob_series = []

for order_seed in ORDER_SEEDS:
    print(f"\nordering seed {order_seed}")
    full_s, Xs, ys, dts, keys, Xts, ids, cols = build_dataset(
        k=BEST_K, order_seed=order_seed
    )

    i_pre = np.flatnonzero(dts < HOLDOUT_START)
    i_hold = np.flatnonzero(dts >= HOLDOUT_START)

    _, p = fit_predict(Xs.iloc[i_pre], ys[i_pre], Xs.iloc[i_hold])
    rep = report_binary(ys[i_hold], p)

    key_hold = keys[i_hold]
    y_series = pd.Series(ys[i_hold], index=key_hold).sort_index()
    p_series = pd.Series(p, index=key_hold).sort_index()

    holdout_prob_series.append(p_series)

    ORDER_ROWS.append({
        "order_seed": order_seed,
        **rep,
    })

    print(
        f"F1={rep['f1']:.6f} P={rep['precision']:.5f} "
        f"R={rep['recall']:.5f} pos={rep['positive_rate']:.5f}"
    )

    del full_s, Xs, Xts
    gc.collect()

common_idx = holdout_prob_series[0].index
for s in holdout_prob_series[1:]:
    assert common_idx.equals(s.index), "holdout row identity changed across orderings"

P = np.vstack([s.to_numpy() for s in holdout_prob_series])
p_avg = P.mean(axis=0)
y_avg = y_series.loc[common_idx].to_numpy()

avg_rep = report_binary(y_avg, p_avg)

order_df = pd.DataFrame(ORDER_ROWS)
order_df.loc[len(order_df)] = {
    "order_seed": "AVERAGED",
    **avg_rep,
}

mean_abs_disagreement = float(np.mean(np.std(P, axis=0)))
mean_pairwise_abs = float(np.mean([
    np.mean(np.abs(P[i] - P[j]))
    for i in range(len(P))
    for j in range(i + 1, len(P))
]))

order_path = os.path.join(WORK, "stage4_ordering_robustness.csv")
order_df.to_csv(order_path, index=False)

print("\nORDERING ROBUSTNESS")
print(order_df.round(6).to_string(index=False))
print("mean per-row probability std across orderings:", round(mean_abs_disagreement, 7))
print("mean pairwise absolute probability difference:", round(mean_pairwise_abs, 7))
print("saved:", order_path)

## Final training and candidate submissions

Two files are produced:

1. `submission_stage4_bestk_singleorder.csv` — ordering seed 0 only.
2. `submission_stage4_bestk_orderavg.csv` — probabilities averaged across ordering seeds 0, 1, and 2.

The order-averaged file is the primary robustness candidate.

Both use threshold **0.5**.

In [ ]:
def make_submission(prob_by_id, tag):
    prob_by_id = prob_by_id.sort_index()
    assert prob_by_id.index.is_unique
    assert np.isfinite(prob_by_id.to_numpy()).all()

    pred = (prob_by_id.to_numpy() >= THRESHOLD).astype(np.int8)
    pos_rate = float(pred.mean())

    # Loud guard against the earlier all-zero submission bug.
    assert 0.05 < pos_rate < 0.80, f"degenerate prediction rate: {pos_rate:.6f}"

    sub = pd.DataFrame({
        "ID": prob_by_id.index.astype(np.int64),
        TARGET: pred,
    })

    if sample is not None:
        assert len(sub) == len(sample)
        assert set(sub["ID"]) == set(sample["ID"])
        sub = sub.set_index("ID").loc[sample["ID"]].reset_index()

    path = os.path.join(WORK, f"submission_{tag}.csv")
    sub.to_csv(path, index=False)
    print(f"{os.path.basename(path)} | positive rate={pos_rate:.6f}")
    return sub, path


test_prob_by_order = []
feature_importance = None

for order_seed in ORDER_SEEDS:
    print(f"\nfinal full-data model | k={BEST_K} | ordering seed={order_seed}")

    full_s, Xs, ys, dts, keys, Xts, ids, cols = build_dataset(
        k=BEST_K, order_seed=order_seed
    )

    model = CatBoostClassifier(**CB_PARAMS)
    model.fit(Xs[cols], ys)
    p = model.predict_proba(Xts[cols])[:, 1]

    assert np.isfinite(p).all()
    prob_series = pd.Series(p, index=ids).sort_index()
    test_prob_by_order.append(prob_series)

    if order_seed == ORDER_SEEDS[0]:
        feature_importance = pd.DataFrame({
            "feature": cols,
            "importance": model.get_feature_importance(),
        }).sort_values("importance", ascending=False)

    del full_s, Xs, Xts, model
    gc.collect()

id_ref = test_prob_by_order[0].index
for s in test_prob_by_order[1:]:
    assert id_ref.equals(s.index), "test ID alignment changed across orderings"

single_probs = test_prob_by_order[0]
avg_probs = pd.Series(
    np.vstack([s.to_numpy() for s in test_prob_by_order]).mean(axis=0),
    index=id_ref,
)

single_sub, single_path = make_submission(
    single_probs, "stage4_bestk_singleorder"
)
avg_sub, avg_path = make_submission(
    avg_probs, "stage4_bestk_orderavg"
)

np.save(
    os.path.join(WORK, "test_proba_stage4_bestk_orderavg.npy"),
    avg_probs.to_numpy()
)

importance_path = os.path.join(WORK, "stage4_feature_importance.csv")
feature_importance.to_csv(importance_path, index=False)

print("\nTop 20 features:")
print(feature_importance.head(20).round(4).to_string(index=False))
print("\nsaved:", importance_path)

In [ ]:
final_positive_rate = float(avg_sub[TARGET].mean())
ordering_avg_f1 = float(avg_rep["f1"])

summary = {
    "stage": "Stage 4 - robust neighbour-width optimization",
    "k_values_tested": K_VALUES,
    "best_k": BEST_K,
    "selection_rule": "maximize min(forward_mean_f1, strict_holdout_f1), then holdout, forward mean, lower std",
    "threshold": THRESHOLD,
    "best_width_metrics": {
        k: (float(v) if isinstance(v, (np.floating, float, int, np.integer)) else v)
        for k, v in summary_df.iloc[0].to_dict().items()
    },
    "ordering_seeds": ORDER_SEEDS,
    "ordering_averaged_holdout_f1": ordering_avg_f1,
    "mean_probability_std_across_orderings": mean_abs_disagreement,
    "mean_pairwise_abs_probability_difference": mean_pairwise_abs,
    "final_orderavg_positive_rate": final_positive_rate,
    "primary_submission": os.path.basename(avg_path),
    "single_order_submission": os.path.basename(single_path),
    "runtime_minutes": (time.time() - T0) / 60.0,
}

summary_json_path = os.path.join(WORK, "stage4_summary.json")
with open(summary_json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 80)
print("STAGE 4 COMPLETE")
print("=" * 80)
print(json.dumps(summary, indent=2))
print("\nNothing was submitted automatically.")
print("Primary candidate:", os.path.basename(avg_path))
print(f"total runtime: {(time.time() - T0) / 60:.2f} min")